# LP-Diff Super-Resolution Training on Kaggle

This notebook trains the **LP-Diff** diffusion-based super-resolution model on Kaggle.

**Workflow:**
- ✅ Clones repository from GitHub
- ✅ Symlinks Kaggle dataset to the expected data directory
- ✅ Auto-detects GPU and optimises batch size
- ✅ Epoch-based training with validation, checkpointing, and best-model tracking
- ✅ Auto-saves checkpoints for resuming after 12h timeout
- ✅ Visualises SR results and training curves

## Step 1: Clones repository from GitHub

In [ ]:
# Patch for Pillow 12+ removing _Ink
import PIL._typing
if not hasattr(PIL._typing, '_Ink'):
    PIL._typing._Ink = str

In [ ]:
import os
import subprocess
import sys

# Clone the repository
repo_url = "https://github.com/hienlongg/MultiFrame-LPR.git"
repo_path = "/kaggle/working/MultiFrame-LPR"
branch = "fix/LPDiff"  # Change this to switch branches

if not os.path.exists(repo_path):
    print(f"Cloning repository from {repo_url} (branch: {branch})...")
    subprocess.run([
        "git", "clone",
        "-b", branch,
        "--single-branch",
        repo_url,
        repo_path
    ], check=True)
    print("Repository cloned successfully!")
else:
    print(f"Repository already exists at {repo_path}")
    try:
        subprocess.run(["git", "-C", repo_path, "pull", "origin", branch], check=False)
        print(f"Pulled latest changes from {branch}")
    except Exception as e:
        print(f"Could not pull: {e}")

os.chdir(repo_path)

# Install project in editable mode WITHOUT touching dependencies.
# Kaggle already has torch, torchvision, numpy etc. pre-installed with
# CUDA support. Letting pip resolve deps would replace them with generic
# PyPI builds that lack GPU kernels or have ABI mismatches.
print("\nInstalling project (--no-deps to preserve Kaggle's CUDA packages)...")
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", ".", "--no-deps"], check=False)
print("Project installed!")

print(f"\nWorking directory: {os.getcwd()}")
print(f"Git branch: {branch}")

## Step 2: Setup Data Path with Symlink

In [ ]:
# Create symlink from Kaggle input data to expected directory
kaggle_data_path = "/kaggle/input/datasets/ixora78/icpr2026-lrlpr"
local_data_path = os.path.join(repo_path, "data")

# Remove existing symlink/directory if it exists
if os.path.islink(local_data_path):
    os.remove(local_data_path)
    print(f"Removed existing symlink: {local_data_path}")
elif os.path.exists(local_data_path) and os.path.isdir(local_data_path):
    print(f"Warning: {local_data_path} exists as directory (not symlink)")

# Create symlink
if not os.path.exists(local_data_path):
    os.symlink(kaggle_data_path, local_data_path)
    print(f"Symlink created: {local_data_path} -> {kaggle_data_path}")

# Verify
if os.path.islink(local_data_path):
    print("Symlink verified")
    print(f"\nData directory contents:")
    for item in sorted(os.listdir(local_data_path))[:10]:
        print(f"   - {item}")

## Step 3: GPU Detection & Configuration

In [ ]:
import torch
import logging
import warnings
import numpy as np
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

# ── GPU Detection ───────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)

    print(f"GPU: {gpu_name} ({vram_gb:.1f} GB)")

    # Auto-select batch size based on VRAM
    if vram_gb >= 30:
        auto_batch_size = 32
    elif vram_gb >= 14:
        auto_batch_size = 16
    else:
        auto_batch_size = 8
    print(f"Auto batch size: {auto_batch_size}")
else:
    auto_batch_size = 4
    print("No GPU detected — using CPU (very slow for diffusion)")

Device: cpu
No GPU detected — using CPU (very slow for diffusion)


## Step 4: Configuration

Edit the parameters below to customise the training run.

In [ ]:
from configs.config import get_lpdiff_config

cfg = get_lpdiff_config(phase='train')

# ── Device ──────────────────────────────────────────────────
cfg.gpu_ids = [0] if torch.cuda.is_available() else []

# ── Data ────────────────────────────────────────────────────
cfg.train_dataset.dataroot     = os.path.join(local_data_path, 'train')
cfg.train_dataset.height       = 112
cfg.train_dataset.width        = 224
cfg.train_dataset.batch_size   = auto_batch_size
cfg.train_dataset.num_workers  = 4

cfg.val_dataset.dataroot       = cfg.train_dataset.dataroot
cfg.val_dataset.height         = cfg.train_dataset.height
cfg.val_dataset.width          = cfg.train_dataset.width
cfg.val_dataset.batch_size     = 1
cfg.val_dataset.num_workers    = 0

# ── Training ────────────────────────────────────────────────
cfg.train.epochs                = 30
cfg.train.val_freq              = 1      # validate every N epochs
cfg.train.save_checkpoint_freq  = 10     # checkpoint every N epochs
cfg.train.print_freq            = 200    # log every N iterations
cfg.train.optimizer.lr          = 1e-4   # standard DDPM LR (5e-3 was WAY too high)

# ── Diffusion ───────────────────────────────────────────────
cfg.beta_schedule_train.n_timestep = 1000
cfg.beta_schedule_val.n_timestep   = 1000

# ── Paths (Kaggle writable) ────────────────────────────────
OUTPUT_DIR = '/kaggle/working/lpdiff_results'
cfg.path.log        = os.path.join(OUTPUT_DIR, 'logs')
cfg.path.tb_logger  = os.path.join(OUTPUT_DIR, 'tb_logger')
cfg.path.results    = os.path.join(OUTPUT_DIR, 'results')
cfg.path.checkpoint = os.path.join(OUTPUT_DIR, 'checkpoint')

VAL_SPLIT_FILE = os.path.join('/kaggle/working', 'val_tracks.json')
SEED = 42

for d in [cfg.path.log, cfg.path.results, cfg.path.checkpoint, cfg.path.tb_logger]:
    os.makedirs(d, exist_ok=True)

print('Config ready.')
print(f'  Data root:   {cfg.train_dataset.dataroot}')
print(f'  Batch size:  {cfg.train_dataset.batch_size}')
print(f'  Epochs:      {cfg.train.epochs}')
print(f'  LR:          {cfg.train.optimizer.lr}')
print(f'  Timesteps:   {cfg.beta_schedule_train.n_timestep}')
print(f'  Output:      {OUTPUT_DIR}')

## Step 5: Check for Previous Checkpoints (Resume Training)

In [ ]:
import glob
import re

# Scan for existing checkpoints
ckpt_dir = cfg.path.checkpoint
gen_files = sorted(glob.glob(os.path.join(ckpt_dir, '*_gen.pth')))

if gen_files:
    # Parse latest checkpoint: I{iter}_E{epoch}_gen.pth
    latest = gen_files[-1]
    prefix = latest.rsplit('_gen.pth', 1)[0]
    print(f"Found checkpoint to resume from:")
    print(f"   {prefix}")
    print(f"   Size: {os.path.getsize(latest) / 1e6:.1f} MB")

    # Enable resume
    cfg.path.resume_state = prefix
    cfg.train.resume_training = True
else:
    print("No previous checkpoints found. Starting fresh training.")
    cfg.path.resume_state = None
    cfg.train.resume_training = False

## Step 6: Seed, Logging & Datasets

In [ ]:
from src.utils.common import seed_everything
from src.data.lpdiff_dataset import create_dataset, create_dataloader

seed_everything(SEED)

# Setup loggers
for name in ['base', 'val']:
    lg = logging.getLogger(name)
    lg.handlers.clear()
    lg.setLevel(logging.INFO)

logger = logging.getLogger('base')
fh = logging.FileHandler(os.path.join(cfg.path.log, 'train.log'), mode='a')
fh.setFormatter(logging.Formatter('%(asctime)s - %(message)s'))
logger.addHandler(fh)
ch = logging.StreamHandler(sys.stdout)
ch.setFormatter(logging.Formatter('%(message)s'))
logger.addHandler(ch)

val_logger = logging.getLogger('val')
vfh = logging.FileHandler(os.path.join(cfg.path.log, 'val.log'), mode='a')
vfh.setFormatter(logging.Formatter('%(asctime)s - %(message)s'))
val_logger.addHandler(vfh)

# Datasets
train_set = create_dataset(cfg.train_dataset, phase='train',
                           val_split_file=VAL_SPLIT_FILE, seed=SEED)
train_loader = create_dataloader(train_set, cfg.train_dataset, phase='train')

val_set = create_dataset(cfg.val_dataset, phase='val',
                         val_split_file=VAL_SPLIT_FILE, seed=SEED)
val_loader = create_dataloader(val_set, cfg.val_dataset, phase='val')

print(f'Train: {len(train_set)} samples, {len(train_loader)} batches/epoch')
print(f'Val:   {len(val_set)} samples')

# Quick sanity check
sample = train_set[0]
print(f'\nSample keys: {list(sample.keys())}')
for k, v in sample.items():
    if isinstance(v, torch.Tensor):
        print(f'  {k}: {v.shape}  range=[{v.min():.2f}, {v.max():.2f}]')

## Step 7: Build Model

In [ ]:
from src.models.LPDiff.model import DDPM

diffusion = DDPM(cfg)
diffusion.print_network()

total_params = sum(p.numel() for p in diffusion.netG.parameters())
train_params = sum(p.numel() for p in diffusion.netG.parameters() if p.requires_grad)
print(f'Parameters: {total_params:,} total, {train_params:,} trainable')

if cfg.path.resume_state:
    print(f'\nResumed from epoch {diffusion.begin_epoch}, step {diffusion.begin_step}')

## Step 8: Training loop

In [ ]:
from src.utils.metrics import tensor2img, save_img, calculate_psnr


def validate(diffusion, val_loader, cfg, epoch, global_step):
    """Run validation, save sample images, return (avg_psnr, avg_loss)."""
    result_path = os.path.join(cfg.path.results, str(epoch))
    os.makedirs(result_path, exist_ok=True)

    diffusion.set_new_noise_schedule(cfg.beta_schedule_val, schedule_phase='val')

    total_psnr, total_loss, count = 0.0, 0.0, 0
    for val_data in tqdm(val_loader, desc=f'Val E{epoch}', leave=False):
        count += 1
        diffusion.feed_data(val_data)
        loss = diffusion.test(continous=False)
        visuals = diffusion.get_current_visuals()

        sr_img = tensor2img(visuals['SR'])
        hr_img = tensor2img(visuals['HR'])

        save_img(sr_img, os.path.join(result_path, f'E{epoch}_{count}_sr.png'))
        save_img(hr_img, os.path.join(result_path, f'E{epoch}_{count}_hr.png'))
        for i in range(1, 6):
            key = f'LR{i}'
            if key in visuals:
                save_img(tensor2img(visuals[key]),
                         os.path.join(result_path, f'E{epoch}_{count}_lr{i}.png'))

        total_psnr += calculate_psnr(sr_img, hr_img)
        total_loss += loss.item() if hasattr(loss, 'item') else float(loss)

    avg_psnr = total_psnr / max(count, 1)
    avg_loss = total_loss / max(count, 1)

    logger.info(f'# Validation # Epoch {epoch}  PSNR: {avg_psnr:.4f}  Loss: {avg_loss:.4e}')
    val_logger.info(f'<epoch:{epoch:3d}, step:{global_step:8,d}> '
                    f'psnr: {avg_psnr:.4e} loss: {avg_loss:.4e}')
    return avg_psnr, avg_loss


# ── Training ────────────────────────────────────────────────
global_step  = diffusion.begin_step
start_epoch  = diffusion.begin_epoch
total_epochs = cfg.train.epochs
best_loss    = float('inf')
best_psnr    = float('-inf')

diffusion.set_new_noise_schedule(cfg.beta_schedule_train, schedule_phase='train')

history = {'epoch': [], 'train_loss': [], 'val_psnr': [], 'val_loss': []}

logger.info(f'Training for {total_epochs} epochs '
            f'(resume from epoch {start_epoch}, step {global_step})')

for epoch in range(start_epoch + 1, total_epochs + 1):
    epoch_loss = 0.0
    num_batches = 0

    pbar = tqdm(train_loader, desc=f'Epoch {epoch}/{total_epochs}', leave=True)
    for train_data in pbar:
        global_step += 1
        num_batches += 1

        diffusion.feed_data(train_data)
        diffusion.optimize_parameters()

        logs = diffusion.get_current_log()
        l_pix = logs.get('l_pix', 0.0)
        epoch_loss += l_pix

        pbar.set_postfix(l_pix=f'{l_pix:.4f}')

        # Per-iteration logging
        if global_step % cfg.train.print_freq == 0:
            msg = f'<epoch:{epoch:3d}, iter:{global_step:8,d}> '
            for k, v in logs.items():
                msg += f'{k}: {v:.4e} '
            logger.info(msg)

    # ── End-of-epoch ──
    avg_epoch_loss = epoch_loss / max(num_batches, 1)
    history['epoch'].append(epoch)
    history['train_loss'].append(avg_epoch_loss)

    logger.info(f'Epoch {epoch}/{total_epochs} complete — '
                f'avg l_pix: {avg_epoch_loss:.4e}, steps: {num_batches}')

    # ── Validation ──
    if epoch % cfg.train.val_freq == 0:
        avg_psnr, avg_loss = validate(
            diffusion, val_loader, cfg, epoch, global_step)

        history['val_psnr'].append(avg_psnr)
        history['val_loss'].append(avg_loss)

        # Best-model checkpointing
        if avg_loss <= best_loss and avg_psnr >= best_psnr:
            best_loss, best_psnr = avg_loss, avg_psnr
            diffusion.save_best_both(epoch, global_step)
        elif avg_loss < best_loss:
            best_loss = avg_loss
            diffusion.save_best_loss(epoch, global_step)
        elif avg_psnr > best_psnr:
            best_psnr = avg_psnr
            diffusion.save_best_psnr(epoch, global_step)

        # Restore training schedule
        diffusion.set_new_noise_schedule(
            cfg.beta_schedule_train, schedule_phase='train')

    # ── Periodic checkpoint ──
    if epoch % cfg.train.save_checkpoint_freq == 0:
        logger.info('Saving checkpoint...')
        diffusion.save_network(epoch, global_step)

logger.info('Training complete.')
print(f'\nDone — {total_epochs} epochs, {global_step} total steps.')

## Step 9: Training Curves

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(history['epoch'], history['train_loss'])
axes[0].set_title('Train Loss (l_pix)')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('L1 Loss')

val_epochs = [e for e in history['epoch'] if e % cfg.train.val_freq == 0]
if history['val_psnr']:
    axes[1].plot(val_epochs[:len(history['val_psnr'])], history['val_psnr'], 'g-o')
    axes[1].set_title('Val PSNR (dB)')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('PSNR')

if history['val_loss']:
    axes[2].plot(val_epochs[:len(history['val_loss'])], history['val_loss'], 'r-o')
    axes[2].set_title('Val Loss (MSE)')
    axes[2].set_xlabel('Epoch')
    axes[2].set_ylabel('MSE')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'training_curves.png'), dpi=150)
plt.show()

## Step 10: Visualise SR Results

Show a few validation samples from the last validation epoch.

In [ ]:
import cv2

results_dir = cfg.path.results
result_dirs = sorted([d for d in os.listdir(results_dir)
                      if os.path.isdir(os.path.join(results_dir, d))],
                     key=lambda x: int(x) if x.isdigit() else 0)
if result_dirs:
    last_dir = os.path.join(results_dir, result_dirs[-1])
    sr_files = sorted([f for f in os.listdir(last_dir) if f.endswith('_sr.png')])[:4]

    if sr_files:
        fig, axes = plt.subplots(len(sr_files), 4, figsize=(16, 4 * len(sr_files)))
        if len(sr_files) == 1:
            axes = [axes]

        for row, sr_f in enumerate(sr_files):
            prefix = sr_f.rsplit('_sr.png', 1)[0]
            imgs = {}
            for suffix in ['lr1', 'lr3', 'sr', 'hr']:
                p = os.path.join(last_dir, f'{prefix}_{suffix}.png')
                if os.path.exists(p):
                    img = cv2.imread(p)
                    imgs[suffix] = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

            for col, key in enumerate(['lr1', 'lr3', 'sr', 'hr']):
                ax = axes[row][col]
                if key in imgs:
                    ax.imshow(imgs[key])
                ax.set_title(key.upper())
                ax.axis('off')

        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR, 'sr_comparison.png'), dpi=150)
        plt.show()
    else:
        print('No SR images found in the latest results folder.')
else:
    print('No validation results found yet.')

## Step 11: Package Results for Kaggle Output

Create a zip archive of checkpoints and results so they persist as Kaggle output artifacts.

In [ ]:
import shutil

kaggle_output = '/kaggle/working'

# Copy best checkpoint to Kaggle output root for easy download
ckpt_dir = cfg.path.checkpoint
if os.path.isdir(ckpt_dir):
    ckpts = sorted([f for f in os.listdir(ckpt_dir) if f.endswith('.pth')])
    if ckpts:
        best = ckpts[-1]
        shutil.copy2(os.path.join(ckpt_dir, best), os.path.join(kaggle_output, best))
        print(f'Copied {best} to {kaggle_output}')

# Zip all results
results_zip = os.path.join(kaggle_output, 'lpdiff_results')
if os.path.isdir(OUTPUT_DIR):
    shutil.make_archive(results_zip, 'zip', OUTPUT_DIR)
    print(f'Archived results to {results_zip}.zip')

# Copy training curves
for fname in ['training_curves.png', 'sr_comparison.png']:
    src = os.path.join(OUTPUT_DIR, fname)
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(kaggle_output, fname))

print('Done! Files available in /kaggle/working/ for download.')